# Search-R1 (2025)
---
[[paper]](https://arxiv.org/abs/2501.18585)<br>
Search-R1 = Search-enhanced Reasoning (R1-style)

__Search-R1__ — это метод обучения больших языковых моделей (LLM) для решения сложных задач поиска и рассуждения, который интегрирует механизмы Chain of Thought (CoT) и Reinforcement Learning (RL) непосредственно в процесс взаимодействия с поисковыми системами. 

__Постановка задачи__<br>
Решается задача Agentic Search в рамках RAG-систем. Модель должна не просто выдать ответ на основе предоставленного контекста, а самостоятельно решить: нужно ли обращаться к поисковику, как сформулировать запрос, как интерпретировать результаты и нужно ли проводить итеративный поиск для уточнения деталей.

__Мотивация__<br>
Современные RAG-системы и AI-агенты часто сталкиваются с проблемой "галлюцинаций в планировании". Модели либо избыточно используют поиск там, где это не нужно, либо не могут декомпозировать сложный многошаговый запрос (Multi-hop query) на серию простых поисковых действий. Традиционное дообучение через Supervised Fine-Tuning (SFT) ограничено качеством обучающих данных, которые сложно собрать для многоступенчатых рассуждений. Необходимо решение, которое позволит модели самой "нащупать" оптимальную стратегию поиска через систему вознаграждений.

__Существующие подходы__<br>
На момент появления Search-R1 существовали следующие подходы:
- RAG (2020): статический подход, где поиск происходит один раз перед генерацией. Не учитывает необходимость уточняющих запросов.
- Self-RAG (2023): использует специальные Reflection Tokens для оценки необходимости поиска. Обучается через SFT, что ограничивает глубину рассуждений качеством разметки.
- ReAct (2022): объединяет рассуждения и действия (Reason + Act). Однако без специфического RL-обучения такие модели часто зацикливаются или выдают нерелевантные поисковые запросы при столкновении с двусмысленностью.
- DeepSeek-R1 (2024/2025): показал эффективность RL (GRPO) для математических и логических задач, но изначально не был оптимизирован для работы с внешними инструментами поиска.

__Идея__<br>
Авторы предложили перенести успех RL-методов (в частности, алгоритма GRPO) из области математики в область информационного поиска. Основная идея заключается в том, чтобы обучать модель рассуждать в формате CoT перед выполнением поискового действия и после получения результатов. Модель получает награду не только за правильный итоговый ответ, но и за релевантность сгенерированных поисковых запросов и эффективность использования найденной информации.

__Архитектура__<br>
Search-R1 базируется на стандартной архитектуре Transformer (например, Llama-3 или Qwen-2.5) и включает следующие компоненты:
1.  LLM Backbone: основная модель, генерирующая скрытые рассуждения (Thought) и специальные теги действий (например, `<search>query</search>`).
2.  Environment (Search Engine): внешний API поисковой системы, который возвращает сниппеты документов по запросу модели.
3.  Reasoning Controller: логика, которая приостанавливает генерацию текста при обнаружении тега поиска, делает запрос к API и вставляет результат в контекст модели в виде тега `<observation>`.

__Алгоритм обучения__<br>
Процесс обучения разделен на два ключевых этапа:
1.  Cold-Start (SFT): Модель дообучается на небольшом наборе высококачественных траекторий "Вопрос -> Рассуждение -> Поиск -> Ответ", чтобы она выучила синтаксис тегов и базовую логику использования инструментов.
2.  RL Stage (GRPO): Основной этап. Используется Group Relative Policy Optimization (GRPO), который позволяет оптимизировать политику модели без использования отдельной модели-критика (Value Function), что экономит до 50% видеопамяти. 
    - Награды (Rewards):
        - Accuracy Reward: +1 за правильный финальный ответ.
        - Format Reward: награда за правильное использование тегов `<thought>` и `<search>`.
        - Consistency Reward: сравнение ответа, полученного с поиском и без него, для минимизации лишних вызовов API.

__Алгоритм инференса__<br>
1.  User Query: Модель получает вопрос.
2.  Internal Monologue: Модель начинает генерировать CoT внутри тега `<thought>`, планируя шаги.
3.  Action: Если модель решает, что данных не хватает, она генерирует `<search>keyword</search>`.
4.  Observation: Система выполняет поиск и подставляет результаты в контекст.
5.  Refinement: Модель анализирует результаты, может выполнить еще один поиск или перейти к итоговому ответу.
6.  Final Response: Формирование ответа пользователю.

__Результаты__<br>
Метод тестировался на бенчмарках для сложных вопросов (HotpotQA, MuSiQue):
- Точность (F1-score) на многоходовых задачах выросла на 12пп по сравнению с базовой моделью, обученной только через SFT.
- Search-R1 сократил количество нерелевантных поисковых запросов на 25% благодаря штрафам в RL за избыточность, что напрямую снижает стоимость эксплуатации системы (Token/API cost).
- Модель продемонстрировала способность к Self-Correction: в 15% случаев модель меняла свое первоначальное мнение после анализа результатов поиска, зафиксированных в CoT.

## 📝 Критический анализ

```markdown
# Search-R1 (2025)
---
[[paper]](https://arxiv.org/abs/2501.18585)<br>
Search-R1 = Search-enhanced Reasoning (R1-style)

__Search-R1__ — метод обучения больших языковых моделей (LLM) для решения сложных задач поиска и рассуждения, интегрирующий Chain of Thought (CoT) и Reinforcement Learning (RL) в процесс взаимодействия с поисковыми системами.

__Постановка задачи__<br>
Решается задача Agentic Search в RAG-системах. Модель должна самостоятельно решать, когда и как использовать поиск, интерпретировать результаты и проводить итеративный поиск для уточнения.

__Мотивация__<br>
Современные RAG-системы сталкиваются с "галлюцинациями в планировании". Модели либо избыточно используют поиск, либо не могут декомпозировать сложные запросы. Необходимо решение, позволяющее модели "нащупать" оптимальную стратегию через вознаграждения.

__Существующие подходы__<br>
- RAG (2020): статический подход, не учитывающий уточняющие запросы.
- Self-RAG (2023): использует Reflection Tokens, ограничен SFT.
- ReAct (2022): объединяет Reason + Act, но без специфического RL-обучения.
- DeepSeek-R1 (2024/2025): эффективен в математике, но не оптимизирован для поиска.

__Идея__<br>
Авторы предложили использовать успех RL-методов (GRPO) в информационном поиске. Модель обучается рассуждать в формате CoT перед и после поиска, получая награды за релевантность запросов и эффективность использования информации.

__Архитектура__<br>
Search-R1 базируется на Transformer (например, Llama-3 или Qwen-2.5) и включает:
1. LLM Backbone: генерирует скрытые рассуждения и теги действий.
2. Environment (Search Engine): API поисковой системы, возвращающий сниппеты.
3. Reasoning Controller: управляет генерацией текста и запросами к API.

<img src="img/img.png" width=500>

__Алгоритм обучения__<br>
1. Cold-Start (SFT): Дообучение на траекториях "Вопрос -> Рассуждение -> Поиск -> Ответ".
2. RL Stage (GRPO): Оптимизация политики без модели-критика, экономия до 50% видеопамяти.
    - Награды:
        - Accuracy Reward: +1 за правильный ответ.
        - Format Reward: за правильное использование тегов.
        - Consistency Reward: минимизация лишних вызовов API.

__Алгоритм инференса__<br>
1. User Query: Получение вопроса.
2. Internal Monologue: Генерация CoT.
3. Action: Генерация `<search>keyword</search>`, если данных не хватает.
4. Observation: Поиск и подстановка результатов.
5. Refinement: Анализ результатов, возможный повторный поиск.
6. Final Response: Формирование ответа.

__Результаты__<br>
Тестирование на HotpotQA, MuSiQue показало:
- Рост точности (F1-score) на 12пп по сравнению с SFT.
- Сокращение нерелевантных запросов на 25%, снижая стоимость эксплуатации.
- Способность к Self-Correction: в 15% случаев модель меняла мнение после анализа результатов.
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример иллюстрации метода Search-R1 (2025) на Python

# Импортируем необходимые библиотеки
import random

# Определяем основные компоненты архитектуры Search-R1

class LLMBackbone:
    """Основная языковая модель, генерирующая скрытые рассуждения и теги действий."""
    def generate_thought(self, query):
        # Генерация скрытых рассуждений (Chain of Thought)
        return f"<thought>Analyzing query: {query}</thought>"

    def generate_search_query(self, thought):
        # Генерация поискового запроса на основе рассуждений
        return f"<search>{thought.split(': ')[1]}</search>"

class SearchEngine:
    """Внешний API поисковой системы."""
    def search(self, query):
        # Возвращаем фиктивные результаты поиска
        return f"<observation>Results for {query}</observation>"

class ReasoningController:
    """Контроллер, управляющий процессом рассуждений и поиска."""
    def __init__(self):
        self.llm = LLMBackbone()
        self.search_engine = SearchEngine()

    def process_query(self, user_query):
        # Генерация внутреннего монолога
        thought = self.llm.generate_thought(user_query)
        print(thought)

        # Решение о необходимости поиска
        if "search" in thought:
            search_query = self.llm.generate_search_query(thought)
            print(search_query)

            # Выполнение поиска и получение результатов
            observation = self.search_engine.search(search_query)
            print(observation)

            # Анализ результатов и формирование ответа
            final_response = self.refine_response(observation)
            print(final_response)
        else:
            print("No search needed. Providing direct response.")

    def refine_response(self, observation):
        # Анализ результатов поиска и формирование окончательного ответа
        return f"Final response based on {observation}"

# Пример использования системы Search-R1

# Создаем контроллер рассуждений
reasoning_controller = ReasoningController()

# Вводим пользовательский запрос
user_query = "What is the capital of France?"

# Обрабатываем запрос с использованием Search-R1
reasoning_controller.process_query(user_query)

# В этом примере мы иллюстрируем основные этапы алгоритма инференса Search-R1:
# 1. Генерация внутреннего монолога (CoT) с использованием LLMBackbone.
# 2. Принятие решения о необходимости поиска.
# 3. Генерация поискового запроса и выполнение поиска через SearchEngine.
# 4. Анализ результатов и формирование окончательного ответа.
# Этот пример демонстрирует, как Search-R1 интегрирует рассуждения и поиск в единую систему.